[List of Finnish regions by life expectancy](https://en.wikipedia.org/wiki/List_of_Finnish_regions_by_life_expectancy) / 
[Продолжительность жизни в областях Финляндии](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_областях_Финляндии)<br />
Data source: [Life expectancy at birth by sex and region](https://pxdata.stat.fi/PXWeb/pxweb/en/StatFin/StatFin__kuol/statfin_kuol_pxt_12an.px) <i>(Select all > Show table > Save result as… > Tab delimited with heading)</i>

In [2]:
import pandas as pd
import math
import re

import sys
sys.path.append("..")
import mal_moduls_private.mal_total as mal

In [3]:
df = pd.read_csv('data/Life expectancy at birth by sex and region -Finland -2024.csv', encoding='iso8859_10', sep='\t', skiprows=2) \
       .sort_values(by=['Region', 'Year'])  \
       .set_index('Region') \
       .rename(columns={'Year': 'year',
                        'Total Life expectancy at birth, years': 'total',
                        'Males Life expectancy at birth, years': 'male',
                        'Females Life expectancy at birth, years': 'female'},
               index={'WHOLE COUNTRY': 'FINLAND'}) \
      .drop(index=['MA1 MAINLAND FINLAND', 'MA2 ÅLAND'])

df.index = df.index.map(lambda st: st[5:] if st != 'FINLAND' else st)

df.index.name = ''

print(len(df))
df.tail()

660


,year,total,male,female
,,,,
FINLAND,2018-2020,81.77,79.03,84.47
FINLAND,2019-2021,81.83,79.11,84.54
FINLAND,2020-2022,81.60,78.93,84.28
FINLAND,2021-2023,81.52,78.91,84.14
FINLAND,2022-2024,81.65,79.07,84.24


<br />
<br />

In [5]:
# explore sitiation with the whole country
df_country = df.loc[df.index=='FINLAND'].set_index('year').T
df_country.columns.name = ''
df_country.loc[:, '2009-2011':]

,2009-2011,2010-2012,2011-2013,2012-2014,2013-2015,2014-2016,2015-2017,2016-2018,2017-2019,2018-2020,2019-2021,2020-2022,2021-2023,2022-2024
total,80.08,80.29,80.58,80.80,81.09,81.24,81.38,81.46,81.65,81.77,81.83,81.60,81.52,81.65
male,76.79,77.13,77.50,77.84,78.18,78.38,78.55,78.68,78.92,79.03,79.11,78.93,78.91,79.07
female,83.30,83.39,83.58,83.69,83.94,84.04,84.15,84.20,84.34,84.47,84.54,84.28,84.14,84.24


In [6]:
# explore sitiation with Åland Islands
df_country = df.loc[df.index=='Åland'].set_index('year').T
df_country.columns.name = ''
df_country.loc[:, '2009-2011':]

,2009-2011,2010-2012,2011-2013,2012-2014,2013-2015,2014-2016,2015-2017,2016-2018,2017-2019,2018-2020,2019-2021,2020-2022,2021-2023,2022-2024
total,82.26,81.47,81.34,81.66,82.56,82.12,82.35,82.58,83.48,83.27,83.75,83.47,84.04,83.73
male,80.01,79.20,78.62,79.16,80.30,80.07,80.17,80.24,81.20,80.80,81.28,81.17,81.77,81.40
female,84.38,83.74,84.28,84.19,84.82,84.08,84.53,84.91,85.79,85.89,86.30,85.84,86.44,86.12


<br />
<br />

In [8]:
df = pd.concat([df.loc[df.year=='2019-2021'].drop(columns='year'),
                df.loc[df.year=='2020-2022'].drop(columns='year'),
                df.loc[df.year=='2021-2023'].drop(columns='year'),
                df.loc[df.year=='2022-2024'].drop(columns='year')], axis='columns')
df.columns = ['2019-21_t', '2019-21_m', '2019-21_f',
              '2020-22_t', '2020-22_m', '2020-22_f',
              '2021-23_t', '2021-23_m', '2021-23_f',
              '2022-24_t', '2022-24_m', '2022-24_f']

df.sort_values(by=['2019-21_t', '2019-21_m'], ascending=False, inplace=True)
df = pd.concat([df.loc[['FINLAND']], df.drop(index='FINLAND')])

df.head()

,2019-21_t,2019-21_m,2019-21_f,2020-22_t,2020-22_m,2020-22_f,2021-23_t,2021-23_m,2021-23_f,2022-24_t,2022-24_m,2022-24_f
,,,,,,,,,,,,
FINLAND,81.83,79.11,84.54,81.60,78.93,84.28,81.52,78.91,84.14,81.65,79.07,84.24
Åland,83.75,81.28,86.30,83.47,81.17,85.84,84.04,81.77,86.44,83.73,81.40,86.12
Ostrobothnia,83.27,81.24,85.35,83.10,81.05,85.21,83.05,80.86,85.33,83.12,80.91,85.45
Central Ostrobothnia,82.55,80.08,85.10,82.32,79.85,84.90,82.27,80.40,84.13,82.22,80.51,83.91
Pirkanmaa,82.51,80.04,84.85,82.18,79.75,84.52,81.90,79.41,84.34,81.84,79.22,84.42


In [9]:
df.insert(loc=3, column='2019-21_fΔm', value=(df['2019-21_f']-df['2019-21_m']).round(2))
df.insert(loc=4, column='Δ1', value=(df['2020-22_t']-df['2019-21_t']).round(2))

df.insert(loc=8, column='2020-22_fΔm', value=(df['2020-22_f']-df['2020-22_m']).round(2))
df.insert(loc=9, column='Δ2', value=(df['2021-23_t']-df['2020-22_t']).round(2))

df.insert(loc=13, column='2021-23_fΔm', value=(df['2021-23_f']-df['2021-23_m']).round(2))
df.insert(loc=14, column='Δ3', value=(df['2022-24_t']-df['2021-23_t']).round(2))

df.insert(loc=18, column='2022-24_fΔm', value=(df['2022-24_f']-df['2022-24_m']).round(2))
df.insert(loc=19, column='Δ_total', value=(df['2022-24_t']-df['2019-21_t']).round(2))

df

,2019-21_t,2019-21_m,2019-21_f,2019-21_fΔm,Δ1,2020-22_t,2020-22_m,2020-22_f,2020-22_fΔm,Δ2,2021-23_t,2021-23_m,2021-23_f,2021-23_fΔm,Δ3,2022-24_t,2022-24_m,2022-24_f,2022-24_fΔm,Δ_total
,,,,,,,,,,,,,,,,,,,,
FINLAND,81.83,79.11,84.54,5.43,-0.23,81.60,78.93,84.28,5.35,-0.08,81.52,78.91,84.14,5.23,0.13,81.65,79.07,84.24,5.17,-0.18
Åland,83.75,81.28,86.30,5.02,-0.28,83.47,81.17,85.84,4.67,0.57,84.04,81.77,86.44,4.67,-0.31,83.73,81.40,86.12,4.72,-0.02
Ostrobothnia,83.27,81.24,85.35,4.11,-0.17,83.10,81.05,85.21,4.16,-0.05,83.05,80.86,85.33,4.47,0.07,83.12,80.91,85.45,4.54,-0.15
Central Ostrobothnia,82.55,80.08,85.10,5.02,-0.23,82.32,79.85,84.90,5.05,-0.05,82.27,80.40,84.13,3.73,-0.05,82.22,80.51,83.91,3.40,-0.33
Pirkanmaa,82.51,80.04,84.85,4.81,-0.33,82.18,79.75,84.52,4.77,-0.28,81.90,79.41,84.34,4.93,-0.06,81.84,79.22,84.42,5.20,-0.67
Southwest Finland,82.35,79.59,85.05,5.46,-0.21,82.14,79.50,84.74,5.24,-0.20,81.94,79.26,84.60,5.34,0.04,81.98,79.35,84.58,5.23,-0.37
South Ostrobothnia,82.25,79.62,84.95,5.33,-0.42,81.83,79.24,84.49,5.25,-0.45,81.38,78.86,83.99,5.13,-0.22,81.16,78.74,83.68,4.94,-1.09
Uusimaa,82.15,79.41,84.68,5.27,-0.24,81.91,79.23,84.39,5.16,0.03,81.94,79.38,84.30,4.92,0.21,82.15,79.69,84.41,4.72,0.00
Central Finland,81.93,79.23,84.68,5.45,-0.40,81.53,78.74,84.42,5.68,-0.26,81.27,78.54,84.11,5.57,0.16,81.43,78.52,84.48,5.96,-0.50


<br />
<br />

In [11]:
# just for interest, explore results: determine regions with max and min values, and also look at specific regions
mal.min_and_max_values(df[['2019-21_t', 'Δ1', '2020-22_t', 'Δ2', '2021-23_t', 'Δ3', '2022-24_t', 'Δ_total']],
                       row_center=['Åland', 'FINLAND'], max_lng=13)

Number of records: 20


,2019-21_t,Δ1,2020-22_t,Δ2,2021-23_t,Δ3,2022-24_t,Δ_total
max,83.75 -Åland,0.17 -South Karelia,83.47 -Åland,0.57 -Åland,84.04 -Åland,0.31 -Kainuu,83.73 -Åland,0.55 -South Karelia
max_2,83.27 -Ostrobothnia,0.15 -North Karelia,83.1 -Ostrobothnia,0.1 -North Karelia,83.05 -Ostrobothnia,0.3 -South Karelia,83.12 -Ostrobothnia,0.35 -Kymenlaakso
max_3,82.55 -Central Ostr…,0.05 -Kanta-Häme,82.32 -Central Ostr…,0.09 -Kymenlaakso,82.27 -Central Ostr…,0.26 -Kanta-Häme,82.22 -Central Ostr…,0.27 -North Karelia
Åland,– 83.75 –,– -0.28 –,– 83.47 –,– 0.57 –,– 84.04 –,– -0.31 –,– 83.73 –,– -0.02 –
FINLAND,– 81.83 –,– -0.23 –,– 81.6 –,– -0.08 –,– 81.52 –,– 0.13 –,– 81.65 –,– -0.18 –
min_3,80.67 -North Karelia,-0.4 -Central Finl…,80.5 -Kymenlaakso,-0.28 -Pirkanmaa,80.48 -South Savo,-0.06 -Pirkanmaa,80.69 -South Savo,-0.5 -Central Finl…
min_2,80.47 -Kymenlaakso,-0.42 -South Ostrob…,80.46 -South Savo,-0.3 -Päijät-Häme,80.48 -Satakunta,-0.22 -South Ostrob…,80.67 -Satakunta,-0.67 -Pirkanmaa
min,80.45 -Kainuu,-0.5 -Kainuu,79.95 -Kainuu,-0.45 -South Ostrob…,79.87 -Kainuu,-0.31 -Åland,80.18 -Kainuu,-1.09 -South Ostrob…


In [12]:
mal.min_and_max_values(df[['2019-21_m', '2020-22_m', '2021-23_m', '2022-24_m', '2019-21_f', '2020-22_f', '2021-23_f', '2022-24_f']],
                       row_center='FINLAND', max_lng=13)

Number of records: 20


,2019-21_m,2020-22_m,2021-23_m,2022-24_m,2019-21_f,2020-22_f,2021-23_f,2022-24_f
max,81.28 -Åland,81.17 -Åland,81.77 -Åland,81.4 -Åland,86.3 -Åland,85.84 -Åland,86.44 -Åland,86.12 -Åland
max_2,81.24 -Ostrobothnia,81.05 -Ostrobothnia,80.86 -Ostrobothnia,80.91 -Ostrobothnia,85.35 -Ostrobothnia,85.21 -Ostrobothnia,85.33 -Ostrobothnia,85.45 -Ostrobothnia
max_3,80.08 -Central Ostr…,79.85 -Central Ostr…,80.4 -Central Ostr…,80.51 -Central Ostr…,85.1 -Central Ostr…,84.9 -Central Ostr…,84.6 -Southwest Fi…,84.67 -South Karelia
FINLAND,– 79.11 –,– 78.93 –,– 78.91 –,– 79.07 –,– 84.54 –,– 84.28 –,– 84.14 –,– 84.24 –
min_3,77.85 -South Savo,77.49 -Kymenlaakso,77.71 -South Savo,77.81 -Satakunta,83.91 -North Karelia,83.62 -South Savo,83.45 -South Savo,83.51 -South Savo
min_2,77.69 -North Karelia,77.49 -South Savo,77.43 -Kymenlaakso,77.79 -Kymenlaakso,83.71 -Kymenlaakso,83.62 -Satakunta,83.36 -Satakunta,83.47 -Päijät-Häme
min,77.39 -Kymenlaakso,77.47 -Kainuu,77.11 -Kainuu,77.24 -Kainuu,83.2 -Kainuu,82.63 -Kainuu,82.97 -Kainuu,83.27 -Lapland


In [13]:
# just for interest, explore difference between regions with max and min life expectancy
for period in ['2019-21_t', '2020-22_t', '2021-23_t', '2022-24_t']:
    print(f"{period[:-2]}: {(df[[period]].max().iloc[0] - df[[period]].min().iloc[0]):.2f}")

2019-21: 3.30
2020-22: 3.52
2021-23: 4.17
2022-24: 3.55


<br />
<br />

In [15]:
# print('dd_replacement = {')
# for region in sorted(df.index.to_list()):
#     print(f'    "{region}"   : {{"fi": ("", ""), "en": ("", ""), "ru": ("", "")}},')
# print('}')

In [16]:
dd_replacement = {
    "FINLAND"   : {"fi": ("Suomi", ""), "en": ("Finland on average", ""), "ru": ("Финляндия в среднем", "")},
    "Åland"   : {"fi": ("Ahvenanmaan", "Ahvenanmaan maakunta"), "en": ("Åland", "Åland"), "ru": ("Ала́ндские острова", "Аландские острова")},
    "Central Finland"   : {"fi": ("Keski-Suomen", "Keski-Suomen maakunta"), "en": ("Central Finland", "Central Finland"), "ru": ("Центральная Финляндия", "Центральная Финляндия (область)")},
    "Central Ostrobothnia"   : {"fi": ("Keski-Pohjanmaan", "Keski-Pohjanmaan maakunta"), "en": ("Central Ostrobothnia", "Central Ostrobothnia"), "ru": ("Центральная Остробо́тния", "Центральная Остроботния")},
    "Kainuu"   : {"fi": ("Kainuun", "Kainuun maakunta"), "en": ("Kainuu", "Kainuu"), "ru": ("Ка́йнуу", "Кайнуу")},
    "Kanta-Häme"   : {"fi": ("Kanta-Hämeen", "Kanta-Hämeen maakunta"), "en": ("Kanta-Häme", "Kanta-Häme"), "ru": ("Канта-Хяме", "Канта-Хяме")},
    "Kymenlaakso"   : {"fi": ("Kymenlaakson", "Kymenlaakson maakunta"), "en": ("Kymenlaakso", "Kymenlaakso"), "ru": ("Кюменлааксо", "Кюменлааксо")},
    "Lapland"   : {"fi": ("Lapin", "Lapin maakunta"), "en": ("Lapland", "Lapland (Finland)"), "ru": ("Лапла́ндия", "Лапландия (область)")},
    "North Karelia"   : {"fi": ("Pohjois-Karjalan", "Pohjois-Karjalan maakunta"), "en": ("North Karelia", "North Karelia"), "ru": ("Северная Карелия", "Северная Карелия (область)")},
    "North Ostrobothnia"   : {"fi": ("Pohjois-Pohjanmaan", "Pohjois-Pohjanmaan maakunta"), "en": ("North Ostrobothnia", "North Ostrobothnia"), "ru": ("Северная Остробо́тния", "Северная Остроботния")},
    "North Savo"   : {"fi": ("Pohjois-Savon", "Pohjois-Savon maakunta"), "en": ("North Savo", "North Savo"), "ru": ("Северное Са́во", "Северное Саво")},
    "Ostrobothnia"   : {"fi": ("Pohjanmaan", "Pohjanmaan maakunta"), "en": ("Ostrobothnia", "Ostrobothnia (region)"), "ru": ("Остробо́тния", "Остроботния (провинция)")},
    "Pirkanmaa"   : {"fi": ("Pirkanmaan", "Pirkanmaan maakunta"), "en": ("Pirkanmaa", "Pirkanmaa"), "ru": ("Пирканмаа", "Пирканмаа")},
    "Päijät-Häme"   : {"fi": ("Päijät-Hämeen", "Päijät-Hämeen maakunta"), "en": ("Päijät-Häme", "Päijät-Häme"), "ru": ("Пя́йят-Хя́ме", "Пяйят-Хяме")},
    "Satakunta"   : {"fi": ("Satakunnan", "Satakunnan maakunta"), "en": ("Satakunta", "Satakunta"), "ru": ("Са́таку́нта", "Сатакунта")},
    "South Karelia"   : {"fi": ("Etelä-Karjalan", "Etelä-Karjalan maakunta"), "en": ("South Karelia", "South Karelia"), "ru": ("Южная Карелия", "Южная Карелия")},
    "South Ostrobothnia"   : {"fi": ("Etelä-Pohjanmaan", "Etelä-Pohjanmaan maakunta"), "en": ("South Ostrobothnia", "South Ostrobothnia"), "ru": ("Южная Остробо́тния", "Южная Остроботния")},
    "South Savo"   : {"fi": ("Etelä-Savon", "Etelä-Savon maakunta"), "en": ("South Savo", "South Savo"), "ru": ("Южное Са́во", "Южное Саво")},
    "Southwest Finland"   : {"fi": ("Varsinais-Suomen", "Varsinais-Suomen maakunta"), "en": ("Southwest Finland", "Southwest Finland"), "ru": ("Ва́рсинайс-Суо́ми", "Варсинайс-Суоми")},
    "Uusimaa"   : {"fi": ("Uudenmaan", "Uudenmaan maakunta"), "en": ("Uusimaa", "Uusimaa"), "ru": ("У́усимаа", "Уусимаа (область)")}
}

In [17]:
# create code for placing info in Wikipedia
def create_table(df, dd_replacement=dd_replacement, file_header='', lang='en'):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)
    
    def chval(x, prec=2, *, add_par=''):  # change_value
        return f'style="color:silver;{add_par}"| —' if math.isnan(x) else \
               f'style="color:darkgreen;{add_par}"| {x:0.{prec}f}' if x>0 else \
               f'style="color:crimson;{add_par}"| −{-x:0.{prec}f}' if x<0 else \
               f'style="color:darkgray;{add_par}"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2, *, add_par=''):  # change_value
        return f'style="color:silver;{add_par}"| \'\'\'—\'\'\'' if math.isnan(x) else \
               f'style="color:darkgreen;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'style="color:crimson;{add_par}"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'style="color:darkgray;{add_par}"| \'\'\'{x:0.{prec}f}\'\'\''

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]
        
        if ser.name in ["FINLAND"]:
            st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2019-21_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2019-21_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2019-21_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2019-21_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δ1"], add_par="padding-right:1.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2020-22_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2020-22_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2020-22_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2020-22_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δ2"], add_par="padding-right:1.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021-23_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2021-23_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2021-23_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2021-23_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δ3"], add_par="padding-right:1.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2022-24_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2022-24_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2022-24_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2022-24_fΔm"])}\'\'\' ' + \
                  f'||{chval_bold(ser["Δ_total"], add_par="padding-right:1.5ex;border-left-width:2px;")}'
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| {if_value(ser["2019-21_t"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2019-21_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2019-21_f"])} ' + \
                  f'|| {if_value(ser["2019-21_fΔm"])} ' + \
                  f'||{chval(ser["Δ1"], add_par="padding-right:1.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {if_value(ser["2020-22_t"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2020-22_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2020-22_f"])} ' + \
                  f'|| {if_value(ser["2020-22_fΔm"])} ' + \
                  f'||{chval(ser["Δ2"], add_par="padding-right:1.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {if_value(ser["2021-23_t"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2021-23_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2021-23_f"])} ' + \
                  f'|| {if_value(ser["2021-23_fΔm"])} ' + \
                  f'||{chval(ser["Δ3"], add_par="padding-right:1.5ex;border-left-width:2px;")} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {if_value(ser["2022-24_t"])} ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2022-24_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2022-24_f"])} ' + \
                  f'|| {if_value(ser["2022-24_fΔm"])} ' + \
                  f'||{chval(ser["Δ_total"], add_par="padding-right:1.5ex;border-left-width:2px;")}'
            
    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')
               # .replace('padding-right:2,5ex;', 'padding-right:2.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    # st = st.replace(';"|—', ';color:silver;"|—')

    return st


table_code = create_table(df, file_header='Finland_header_ru -2024.txt', lang='ru')

# write the code to file
with open('output/Table code for Finnish regions -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [18]:
table_code = create_table(df, file_header='Finland_header_en -2024.txt', lang='en')

# write the code to file
with open('output/Table code for Finnish regions -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)